# 01 · Random Forest: predicting HbA1c from CGM and clinical features

**Goal:** check whether a non-linear model (Random Forest) can extract more signal than the linear models in `01_linear_model.ipynb`, and whether tuning its hyperparameters helps.

**Protocol:** the same 80/20 train/test split as the linear notebook (92 train, 24 test). Models are compared with 5-fold cross-validation on the training set. The test set is scored once per experiment and is used to compare experiments, not to tune them.

**Experiments**
1. Random Forest with default settings, 5 features
2. Grid search on the same 5 features
3. Random Forest with default settings, 20 features
4. Grid search on the same 20 features

The feature sets match the linear notebook, so the results can be compared directly.

## 1. Setup

In [1]:
# Make the project root importable so `src` can be imported from this notebook
from pathlib import Path
import sys

FILE = Path.cwd().parent.resolve()
sys.path.insert(0, str(FILE))

In [2]:
from src.prepare_splits import load_splits
from src.data_utils import get_Xy

from sklearn.model_selection import cross_validate, GridSearchCV
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.ensemble import RandomForestRegressor
import numpy as np
import pandas as pd

## 2. Data

Same splits as the linear notebook: 92 patients for training and 24 for testing. The `patient` column is an identifier and is never used as a feature.

In [3]:
# Load the train/test splits saved by src/prepare_splits.py
splits = load_splits()
splits.keys()

dict_keys(['train', 'test'])

In [4]:
# Check split sizes and available columns
for split_name, data in splits.items():
    print(f'Split: {split_name}, n_samples: {len(data)}')
    print(data.columns)

Split: train, n_samples: 92
Index(['patient', 'tar', 'tir', 'tbr', 'hba1c', 'age', 'gender', 'bmi',
       'height', 'weight', 'smoking_history', 'alcohol_drinker',
       'diabetes_type', 'diabetes_duration', 'cgm_mean', 'cgm_std', 'cgm_min',
       'cgm_max', 'cgm_cv', 'cgm_q25', 'cgm_q75', 'cgm_iqr',
       'glucose_variability', 'n_hypo_events', 'n_hyper_events',
       'rapid_changes', 'mean_excursion', 'pc1', 'pc2', 'pc3'],
      dtype='str')
Split: test, n_samples: 24
Index(['patient', 'tar', 'tir', 'tbr', 'hba1c', 'age', 'gender', 'bmi',
       'height', 'weight', 'smoking_history', 'alcohol_drinker',
       'diabetes_type', 'diabetes_duration', 'cgm_mean', 'cgm_std', 'cgm_min',
       'cgm_max', 'cgm_cv', 'cgm_q25', 'cgm_q75', 'cgm_iqr',
       'glucose_variability', 'n_hypo_events', 'n_hyper_events',
       'rapid_changes', 'mean_excursion', 'pc1', 'pc2', 'pc3'],
      dtype='str')


## 3. Baseline: mean-only predictor

**Goal:** a floor for comparison. This model ignores all features and predicts the training mean HbA1c for every test patient. Any useful model has to beat it.

On the training data its R² is 0 by construction. On new data it is slightly negative, because the test mean differs from the training mean.

In [5]:
# Mean-only predictor on the saved split (no features used)
from sklearn.dummy import DummyRegressor

y_train = splits['train']['hba1c']
y_test = splits['test']['hba1c']

dummy = DummyRegressor(strategy='mean')
cv_dummy = cross_validate(
    dummy, splits['train'][['tar']], y_train,
    cv=5,
    scoring=['r2', 'neg_mean_squared_error', 'neg_mean_absolute_error'],
    return_train_score=True
)
dummy.fit(splits['train'][['tar']], y_train)
y_pred_test = dummy.predict(splits['test'][['tar']])

print(f"Val R²: {cv_dummy['test_r2'].mean():.4f} (+/- {cv_dummy['test_r2'].std():.4f})")
print(f"Val RMSE: {np.sqrt(-cv_dummy['test_neg_mean_squared_error'].mean()):.4f}")
print(f"Val MAE: {-cv_dummy['test_neg_mean_absolute_error'].mean():.4f}")
print(f"\nTest R²: {r2_score(y_test, y_pred_test):.4f}")
print(f"Test RMSE: {np.sqrt(mean_squared_error(y_test, y_pred_test)):.4f}")
print(f"Test MAE: {mean_absolute_error(y_test, y_pred_test):.4f}")

Val R²: -0.1037 (+/- 0.0897)
Val RMSE: 26.9910
Val MAE: 21.8510

Test R²: -0.0374
Test RMSE: 30.2985
Test MAE: 25.9825


## 4. Evaluation helpers

Two helpers run the experiments:
- `fit_random_forest` fits one forest with fixed hyperparameters. It reports 5-fold CV on the training set, a single test score, and the top 5 features by importance.
- `rf_grid_search` searches a grid of hyperparameters with 5-fold CV on the training set, then scores the best forest on the test set.

In [6]:
def fit_random_forest(splits=splits, 
                      x_cols=['tar', 'tbr'], 
                      y_col='hba1c', 
                      degree=1, 
                      n_estimators=100,
                      max_depth=10,
                      min_samples_split=5,
                      min_samples_leaf=2):
    """
    Fit a Random Forest with fixed hyperparameters and report CV and test metrics.

    Args:
        splits: Dictionary with 'train' and 'test' DataFrames.
        x_cols: Feature column names.
        y_col: Target column name.
        degree: Polynomial degree applied to the features (1 means none).
        n_estimators, max_depth, min_samples_split, min_samples_leaf: Forest settings.

    Returns:
        The RandomForestRegressor fitted on the full training split.
    """

    
    X_train, y_train, X_test, y_test = get_Xy(splits=splits, x_cols=x_cols, y_col=y_col, degree=degree)

    # Create Random Forest model
    rf_model = RandomForestRegressor(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        min_samples_leaf=min_samples_leaf,
        random_state=42,
        n_jobs=-1
    )

    # Cross-validation on train set
    cv_scores = cross_validate(
        rf_model, X_train, y_train,
        cv=5,
        scoring=['r2', 'neg_mean_squared_error', 'neg_mean_absolute_error'],
        return_train_score=True
    )

    print(f"=== RANDOM FOREST ===")
    print(f"Train R²: {cv_scores['train_r2'].mean():.4f} (+/- {cv_scores['train_r2'].std():.4f})")
    print(f"Val R²: {cv_scores['test_r2'].mean():.4f} (+/- {cv_scores['test_r2'].std():.4f})")
    print(f"Val RMSE: {np.sqrt(-cv_scores['test_neg_mean_squared_error'].mean()):.4f}")
    print(f"Val MAE: {-cv_scores['test_neg_mean_absolute_error'].mean():.4f}")

    # Fit on entire train set and evaluate on test set
    rf_model.fit(X_train, y_train)
    y_pred_test = rf_model.predict(X_test)

    print(f"\nTest R²: {r2_score(y_test, y_pred_test):.4f}")
    print(f"Test RMSE: {np.sqrt(np.mean((y_test - y_pred_test)**2)):.4f}")
    print(f"Test MAE: {np.mean(np.abs(y_test - y_pred_test)):.4f}")

    # Feature importance
    feature_importance = pd.DataFrame({
        'feature': X_train.columns,
        'importance': rf_model.feature_importances_
    }).sort_values('importance', ascending=False)

    print(f"\nTop 5 Important Features:")
    print(feature_importance.head())

    return rf_model

## 5. Experiment 1: default Random Forest, 5 features

**Goal:** a first look at the forest on the same five features used in experiment 3 of the linear notebook: TAR, TBR, age, gender and diabetes type.

In [7]:
# Experiment 1: default forest, 5 features
fit_random_forest(splits=splits, x_cols=['tar', 'tbr', 'age', 'gender', 'diabetes_type'], y_col='hba1c')

=== RANDOM FOREST ===
Train R²: 0.7289 (+/- 0.0142)
Val R²: -0.0274 (+/- 0.2411)
Val RMSE: 25.0198
Val MAE: 19.5964



Test R²: 0.1710
Test RMSE: 27.0854
Test MAE: 20.0855

Top 5 Important Features:
         feature  importance
0            tar    0.497071
2            age    0.272031
1            tbr    0.168155
3         gender    0.037230
4  diabetes_type    0.025513


,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",10
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",5
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",2
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel. :meth:`fit`, :meth:`predict`,:meth:`decision_path` and :meth:`apply` are all parallelized over thetrees. ``None`` means 1 unless in a :obj:`joblib.parallel_backend`context. ``-1`` means using all processors. See :term:`Glossary<n_jobs>` for more details.",-1
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""squared_error"", ""absolute_error"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""absolute_error"" for the meanabsolute error, which minimizes the L1 loss using the median of each terminalnode, and ""poisson"" which uses reduction in Poisson deviance to find splits,also using the mean of each terminal node... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion... versionchanged:: 1.9 Criterion `""friedman_mse""` was deprecated.",'squared_error'
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"max

In [8]:
def rf_grid_search(param_grid, splits=splits, 
                      x_cols=['tar', 'tbr'], 
                      y_col='hba1c', 
                      degree=1):
    """
    Grid-search Random Forest hyperparameters with 5-fold CV on the train split.

    Args:
        param_grid: Dictionary of hyperparameter names to candidate values.
        splits: Dictionary with 'train' and 'test' DataFrames.
        x_cols: Feature column names.
        y_col: Target column name.
        degree: Polynomial degree applied to the features (1 means none).

    Returns:
        The best forest found by the search, refitted on the full training split.
    """
    # Define base model and parameter grid
    base_model = RandomForestRegressor(random_state=42, n_jobs=-1)

    grid_search = GridSearchCV(
        estimator=base_model,
        param_grid=param_grid,
        cv=5,
        scoring='r2',
        n_jobs=-1,
        verbose=1
    )

    X_train, y_train, X_test, y_test = get_Xy(splits=splits, x_cols=x_cols, y_col=y_col, degree=degree)

    # Fit and search
    grid_search.fit(X_train, y_train)

    # Best parameters
    print(f"Best parameters: {grid_search.best_params_}")
    print(f"Best CV R²: {grid_search.best_score_:.4f}")

    best_model = grid_search.best_estimator_
    y_pred_test = best_model.predict(X_test)

    print(f"Test R²: {r2_score(y_test, y_pred_test):.4f}")
    print(f"Test RMSE: {np.sqrt(np.mean((y_test - y_pred_test)**2)):.4f}")

    return best_model

    

## 6. Experiment 2: grid search, 5 features

**Goal:** check whether tuning the forest on the same five features improves the cross-validated score.

**Search space:** 160 combinations of `max_depth`, `min_samples_split`, `min_samples_leaf` and `max_features`, each with 5-fold CV.

In [9]:
# Experiment 2: grid searched on the 5-feature set
param_grid = {
        'max_depth': [5, 8, 10, 15, 20],
        'min_samples_split': [2, 5, 10, 15],
        'min_samples_leaf': [1, 2, 4, 8],
        'max_features': ['sqrt', 'log2']
    }

rf_grid_search(param_grid=param_grid, splits=splits, x_cols=['tar', 'tbr', 'age', 'gender', 'diabetes_type'], y_col='hba1c')

Fitting 5 folds for each of 160 candidates, totalling 800 fits


Best parameters: {'max_depth': 5, 'max_features': 'log2', 'min_samples_leaf': 1, 'min_samples_split': 2}
Best CV R²: 0.0403
Test R²: 0.1072
Test RMSE: 28.1076


,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",5
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'log2'
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel. :meth:`fit`, :meth:`predict`,:meth:`decision_path` and :meth:`apply` are all parallelized over thetrees. ``None`` means 1 unless in a :obj:`joblib.parallel_backend`context. ``-1`` means using all processors. See :term:`Glossary<n_jobs>` for more details.",-1
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""squared_error"", ""absolute_error"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""absolute_error"" for the meanabsolute error, which minimizes the L1 loss using the median of each terminalnode, and ""poisson"" which uses reduction in Poisson deviance to find splits,also using the mean of each terminal node... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion... versionchanged:: 1.9 Criterion `""friedman_mse""` was deprecated.",'squared_error'
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"m

## 7. Experiment 3: default Random Forest, 20 features

**Goal:** the same 20 features as experiment 4 of the linear notebook, with default settings. The question is whether the forest can use the extra CGM and clinical features where the linear model failed.

In [10]:
# Experiment 3: the 20-feature set, same as the linear notebook
x_cols=['tar', 'tbr', 'age', 'gender', 'bmi',
       'diabetes_type', 'diabetes_duration', 'cgm_mean', 'cgm_std', 'cgm_min',
       'cgm_max', 'cgm_cv', 'cgm_q25', 'cgm_q75', 'cgm_iqr',
       'glucose_variability', 'n_hypo_events', 'n_hyper_events',
       'rapid_changes', 'mean_excursion']

In [11]:
fit_random_forest(splits=splits, x_cols=x_cols, y_col='hba1c')

=== RANDOM FOREST ===
Train R²: 0.8352 (+/- 0.0110)
Val R²: 0.1622 (+/- 0.2355)
Val RMSE: 22.7281
Val MAE: 17.6537



Test R²: 0.1774
Test RMSE: 26.9800
Test MAE: 20.5584

Top 5 Important Features:
              feature  importance
10            cgm_max    0.184535
18      rapid_changes    0.142133
8             cgm_std    0.110337
6   diabetes_duration    0.100528
7            cgm_mean    0.063110


,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",10
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",5
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",2
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel. :meth:`fit`, :meth:`predict`,:meth:`decision_path` and :meth:`apply` are all parallelized over thetrees. ``None`` means 1 unless in a :obj:`joblib.parallel_backend`context. ``-1`` means using all processors. See :term:`Glossary<n_jobs>` for more details.",-1
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""squared_error"", ""absolute_error"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""absolute_error"" for the meanabsolute error, which minimizes the L1 loss using the median of each terminalnode, and ""poisson"" which uses reduction in Poisson deviance to find splits,also using the mean of each terminal node... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion... versionchanged:: 1.9 Criterion `""friedman_mse""` was deprecated.",'squared_error'
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"max

## 8. Experiment 4: grid search, 20 features

**Goal:** tune the forest on the full feature set, using the same search space as experiment 2.

In [12]:
# Experiment 4: grid search on the 20-feature set
rf_grid_search(param_grid=param_grid, splits=splits, x_cols=x_cols, y_col='hba1c')

Fitting 5 folds for each of 160 candidates, totalling 800 fits


Best parameters: {'max_depth': 5, 'max_features': 'log2', 'min_samples_leaf': 1, 'min_samples_split': 5}
Best CV R²: 0.2018
Test R²: 0.1792
Test RMSE: 26.9510


,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",5
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",5
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'log2'
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel. :meth:`fit`, :meth:`predict`,:meth:`decision_path` and :meth:`apply` are all parallelized over thetrees. ``None`` means 1 unless in a :obj:`joblib.parallel_backend`context. ``-1`` means using all processors. See :term:`Glossary<n_jobs>` for more details.",-1
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""squared_error"", ""absolute_error"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""absolute_error"" for the meanabsolute error, which minimizes the L1 loss using the median of each terminalnode, and ""poisson"" which uses reduction in Poisson deviance to find splits,also using the mean of each terminal node... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion... versionchanged:: 1.9 Criterion `""friedman_mse""` was deprecated.",'squared_error'
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"m

## 9. Summary

| Experiment | Features | CV validation R² | Test R² | Test RMSE | Test MAE |
|---|---|---|---|---|---|
| Mean-only predictor | none | -0.104 ± 0.090 | -0.037 | 30.30 | 25.98 |
| 1. Default | 5 | -0.027 ± 0.241 | 0.171 | 27.09 | 20.09 |
| 2. Grid search | 5 | 0.040 (best CV) | 0.107 | 28.11 | not reported |
| 3. Default | 20 | 0.162 ± 0.236 | 0.177 | 26.98 | 20.56 |
| 4. Grid search | 20 | 0.202 (best CV) | 0.179 | 26.95 | not reported |

For comparison, the linear models reached test R² between 0.07 and 0.12 and a validation R² of 0.06 at best (see `01_linear_model.ipynb`). Both notebooks use `r2_score`.

### Observations
- **Mean-only predictor (test R² -0.04, validation -0.10):** all four forest experiments beat it on both validation and test. Even the 5-feature forest, which overfits, improves on the mean (validation -0.03 against -0.10).
- **Default forest on 5 features overfits:** train R² is 0.73 but validation R² is -0.03. Its test R² (0.17) is higher than its validation R², which is a sign that the 24-patient test set is noisy.
- **More features help the forest's validation score**, from -0.03 to 0.16 with default settings. The linear models went the other way.
- **Grid search on 20 features** moves the forest towards shallower trees (`max_depth` 5) and improves the best CV score (0.16 → 0.20), but the test score stays the same (0.18).
- **Grid search on 5 features** raises the CV score (-0.03 → 0.04), but the test score drops (0.17 → 0.11). Tuning on 92 patients picks settings that do not transfer.
- **The most important features are CGM variability and range**: `cgm_max`, `rapid_changes`, `cgm_std`, `diabetes_duration`, `cgm_mean`. For the 5-feature model, TAR and age matter most.
- Our features are weakly predictive, which we already knew from the correlations.

### Conclusions
- **The forest does somewhat better than the linear models on the test set**: test R² of 0.17 to 0.18, against 0.07 to 0.12 for the linear models. With 20 features its validation score is also much better (0.16 against -0.45).
- **The gain is modest and uncertain.** A test R² of about 0.18 on 24 patients is within the noise.
- **Grid search is not worth it here.** It improves the CV score on the full feature set, but not the test score, and on the 5-feature set it hurts the test score. The default settings do as well.

### Next steps
- Repeat the splits over several seeds, as in `03_models_w_pca_features.ipynb`, to estimate how much of the difference is noise.